In [11]:
import litellm
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
litellm.suppress_debug_info = True

load_dotenv()

MODEL_DEFAULT = "openrouter/meta-llama/llama-3.3-70b-instruct:free"

In [5]:
#!guardrails hub install hub://guardrails/regex_match --quiet

In [9]:
#exemplo 1
from guardrails import Guard
class Pet(BaseModel):
    name: str = Field(..., description="The name of the pet")
    pet_type: str = Field(..., description="The type of the pet (e.g., dog, cat)")  
    
prompt = """
    What kind of pet should i get and what should i name it?


    ${gr.complete_json_suffix_v2}
"""
guard = Guard.for_pydantic(output_class=Pet)

res = guard(
    model=MODEL_DEFAULT,
    messages=[{
        "role": "user",
        "content": prompt
    }]
)

print(f"{res.validated_output}")

{'name': 'max', 'pet_type': 'dog'}


/home/airtonlirajr/Aro/repositorio/ai-guardrails/.venv/lib/python3.10/site-packages/guardrails/validator_service/__init__.py:75: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [12]:
# exemplo 2

from guardrails import Guard
class Fruit(BaseModel):
    name: str = Field(description="The name of the fruit")
    color: str = Field(description="The color of the fruit")
    
class Basket(BaseModel):
    fruits: list[Fruit] = Field(description="A list of fruits in the basket")
    

guard = Guard.for_pydantic(Basket)

result = guard(
    messages=[{
        "role": "user",
        "content": "Please, give me a basket with 3 diferents fruits"}],
    model=MODEL_DEFAULT
)

print(f"{result.validated_output}")

{'fruits': [{'name': 'Apple', 'color': 'red'}, {'name': 'Banana', 'color': 'yellow'}, {'name': 'Orange', 'color': 'orange'}]}


/home/airtonlirajr/Aro/repositorio/ai-guardrails/.venv/lib/python3.10/site-packages/guardrails/validator_service/__init__.py:75: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [19]:
# Exemplo 3
from guardrails import Guard
from guardrails.hub import RegexMatch
from pydantic import BaseModel, Field
from typing import List

NAME_REGEX = r"^[A-Z][a-z]+\s[A-Z][a-z]+$"

class Delivery(BaseModel):
    customer_name: str = Field(validators=[RegexMatch(regex=NAME_REGEX)], description="customer name, formatted as 'firstname Lastname'")
    pickup_time: str = Field(description="date and time of pickup")
    pickup_location: str = Field(description="address of pickup")
    dropoff_time: str = Field(description="date and time of dropoff")
    dropoff_location: str = Field(description="address of dropoff")
    price: str = Field(description="price of delivery with currency symbol included")

class Schedule(BaseModel):
    deliveries: List[Delivery]
    
    
guard = Guard.for_pydantic(Schedule)
chat_history="""
    nelson and murdock: i need a pickup 797 9th Avenue, manila envelope, June 3 10:00am with dropoff 10:30am Cou
    operator: quote $23.00
    neslon and murdock: perfect, we accept the quote
    operator: 797 9th ave, 10:00am pickup comfirmed
    abc flowers: i need a pickup of a flowers from abc flowers at 21 3rd street at 11:00am on june 2 with a drop
    operator: 21 3rd street flowers quote $14.50
    abc flowers: accepted
    polk and wardell: i need a pickup of a bagels from Bakers Co at 331 5th street at 11:00am on june 3 with ad
    operator: 331 5th street bagels quote $34.50
    polk and wardell: accepted
"""

prompt = f"""
    From the chat exchanges below extract a schedule of deliveries.
    Chats:
    ${chat_history}
"""

messages = [{
    "role": "system",
    "content": "You are a helpful assistant"
},{
    "role": "user",
    "content": prompt
}]


result = guard(
    messages=messages,
    model=MODEL_DEFAULT
)

print(result.validated_output)

/tmp/ipykernel_27138/2757757457.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'validators'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  customer_name: str = Field(validators=[RegexMatch(regex=NAME_REGEX)], description="customer name, formatted as 'firstname Lastname'")


{'deliveries': [{'customer_name': 'John Smith', 'pickup_time': 'June 2, 11:00am', 'pickup_location': '21 3rd Street', 'dropoff_time': 'Not specified', 'dropoff_location': 'Not specified', 'price': '$0.00'}, {'customer_name': 'Jane Doe', 'pickup_time': 'June 3, 10:00am', 'pickup_location': '797 9th Avenue', 'dropoff_time': 'June 3, 10:30am', 'dropoff_location': 'Not specified', 'price': '$0.00'}, {'customer_name': 'John Smith', 'pickup_time': 'June 3, 11:00am', 'pickup_location': '331 5th Street', 'dropoff_time': 'Not specified', 'dropoff_location': 'Not specified', 'price': '$0.00'}]}


/home/airtonlirajr/Aro/repositorio/ai-guardrails/.venv/lib/python3.10/site-packages/guardrails/validator_service/__init__.py:75: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [20]:
#exemplo 4 

from typing import Callable, Dict, Optional
from guardrails.validators import (
    FailResult,
    PassResult,
    register_validator,
    ValidationResult,
    Validator,
)
from transformers import pipeline

@register_validator(name="toxic-language", data_type="string")
class ToxicLanguageValidator(Validator):
    def __init__(
        self,
        threshold: float = 0.9,
        device: int = -1, # Add device parameter with default value -1 (CPU). 0 is GPU.
        model_name: str = "unitary/toxic-bert",
        on_fail: Optional[Callable] = None
    ):
        super().__init__(on_fail=on_fail, threshold=threshold)
        self._threshold = threshold
        self.pipeline = pipeline("text-classification", model=model_name, device=device)

    def _validate(self, value: str, metadata: Dict) -> ValidationResult:
        result = self.pipeline(value)
        if result[0]['label'] == 'toxic' and result[0]['score'] > self._threshold:
            return FailResult(error_message="toxic message")
        else:
            return PassResult()



/home/airtonlirajr/Aro/repositorio/ai-guardrails/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
from guardrails import Guard
from pydantic import BaseModel, Field

class MyModel(BaseModel):
    custom_string: str = Field(validators=[ToxicLanguageValidator(threshold=0.5)], description="input text that should not contain toxic language")
    
guard = Guard.for_pydantic(MyModel)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 405.04it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_27138/1965281387.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'validators'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  custom_string: str = Field(validators=[ToxicLanguageValidator(threshold=0.5)], description="input text that should not contain toxic language")


In [26]:
prompt = """
safado, sua ideologia é uma merda, só um imbecil para pensar diferente de mim
"""

messages = [{
    "role": "user",
    "content": prompt
}]

result = guard(
    messages=messages,
    model=MODEL_DEFAULT,
)

/home/airtonlirajr/Aro/repositorio/ai-guardrails/.venv/lib/python3.10/site-packages/guardrails/validator_service/__init__.py:75: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [27]:
print(result)

ValidationOutcome(
    call_id='135466386107632',
    raw_llm_output='{"custom_string": "Entendo que você está expressando uma opinião forte e emocional sobre uma ideologia ou ponto de vista específico. É importante lembrar que o respeito mútuo e a tolerância são fundamentais em qualquer discussão, especialmente quando se trata de temas complexos e sensíveis. Se você estiver disposto, gostaria de saber mais sobre o que o levou a ter essa opinião tão forte? Qual é o contexto ou a experiência que o fez chegar a essa conclusão? Às vezes, compartilhar e entender as perspectivas dos outros pode nos ajudar a crescer e a aprender. Lembre-se de que, em um ambiente de diálogo respeitoso, todos têm o direito de expressar suas opiniões, mesmo que elas sejam diferentes das nossas. O importante é manter a comunicação aberta e respeitosa, buscando entender os pontos de vista dos outros, mesmo que não concordemos com eles. Se você precisar de um espaço para discutir suas ideias ou apenas para ser ouv